In [58]:
import torch
import torch.nn as nn
torch.manual_seed(42)

In [59]:
d = 512
max_pos = 4096
base = 10000

In [60]:
# total angles 
angles = torch.arange(0, 4, 1) 
m = torch.arange(0, 3, 1) 
m_angles = torch.outer(m, angles)
print(m_angles)

# repeat angles trick
cache = torch.zeros(3, 8)
print(m_angles[:,0::2])
print(m_angles[:,1::2])
cache[:,0::2] = m_angles
cache[:,1::2] = m_angles
print(cache)

tensor([[0, 0, 0, 0],
        [0, 1, 2, 3],
        [0, 2, 4, 6]])
tensor([[0, 0],
        [0, 2],
        [0, 4]])
tensor([[0, 0],
        [1, 3],
        [2, 6]])
tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 1., 2., 2., 3., 3.],
        [0., 0., 2., 2., 4., 4., 6., 6.]])


In [61]:
# broadcast batch dimention

# X = bs, seq_len, d
# pos = seq_len, d
X = torch.zeros( 2, 3, 4)
pos = torch.randn(3, 4) 
result = X + pos
print(result[0,:,:])
print(result[1,:,:])

tensor([[ 0.3367,  0.1288,  0.2345,  0.2303],
        [-1.1229, -0.1863,  2.2082, -0.6380],
        [ 0.4617,  0.2674,  0.5349,  0.8094]])
tensor([[ 0.3367,  0.1288,  0.2345,  0.2303],
        [-1.1229, -0.1863,  2.2082, -0.6380],
        [ 0.4617,  0.2674,  0.5349,  0.8094]])


In [62]:
class RoPE(nn.Module):
    def __init__(self, d = 512, max_pos = 4096, base = 10000.0):
        super().__init__()
        self.d = 512
        
        self.base = base
        self.max_pos = pos

        n_angles = d // 2
        # print(n_anles)
        
        m = torch.arange(0, max_pos, 1)
        k = torch.arange(0, n_angles, 1) 
        angles = 1 /  self.base ** (2 * k / d)
        
        m_angles = torch.outer(m, angles)
        # print(m_angles.shape)

        # cache, sin,cos
        cos = torch.cos(m_angles)
        sin = torch.sin(m_angles)
        self.linear_cos = torch.zeros(max_pos, d)
        self.linear_sin = torch.zeros(max_pos, d)

        # print(self.linear_cos[:, 0::2].shape)
        # print(cos.shape)

        # fast linear sin,cos
        self.linear_cos[:, 0::2] = self.linear_cos[:, 1::2] = cos
        self.linear_sin[:, 0::2] = self.linear_sin[:, 1::2] = sin
        
    def apply_linear_rope(self, X):
        '''
            input: X[bs, seq_len, d]
        '''
        bs, seq_len, d = X.shape

        # assert seq_len <= self.max_pos
        
        X_shift = torch.zeros_like(X)
        X_shift[:, :, 0::2] = -X[:, :, 1::2]
        X_shift[:, :, 1::2] = X[:, :, 0::2]

        Y = self.linear_cos[:seq_len, :] * X + self.linear_sin[:seq_len, :] * X_shift

        return Y

In [63]:
rope = RoPE(d, max_pos, base)

In [64]:
seq_len = 100
bs = 3
Q = torch.randn(bs, seq_len, d)
rope_q = rope.apply_linear_rope(Q)